In [0]:
%pip install pdfplumber

In [0]:
try:
    import pdfplumber
except ImportError:
    print("Missing dependency: pip install pdfplumber")

In [0]:

path = "/Volumes/promotion_princess/framework/cf_pdf"

def get_latest_file(path):
    # List files (non-recursive)
    files = [f for f in dbutils.fs.ls(path) if not f.isDir()]

    if not files:
        return None
    else:
        last = max(files, key=lambda f: f.modificationTime)
        return last.path[5:]


In [0]:

def extract_all_text(pdf_path): 
    pages = {}
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            contents = []
            contents.append(page.extract_text())
            contents.append(page.extract_tables(table_settings={"intersection_x_tolerance": 2, "intersection_y_tolerance": 30, "snap_x_tolerance": 10, "snap_y_tolerance": 8}) or "")
            pages[i] = contents
                  
    return pages, pdf

In [0]:

pdf_path = get_latest_file("/Volumes/promotion_princess/framework/cf_pdf")
pages, pdf = extract_all_text(pdf_path)
length = len(pages)
print(length)

In [0]:
pages, pdf = extract_all_text(pdf_path)
print(pages[8][1])

In [0]:
print(pages[8][0])

In [0]:
print(pages[18][1])

In [0]:
im = pdf.pages[1].to_image(resolution=150)
im.debug_tablefinder(table_settings={"intersection_x_tolerance": 2, "intersection_y_tolerance": 30, "snap_x_tolerance": 10, "snap_y_tolerance": 8})

In [0]:
SECTION_KEYWORDS = {
    "annex4": ["annex 4: data and ai adoption"],
    "annex3": ["annex 3: managed revenue"],
    "annex2b": ["annex 2b: technical skills progression"],
    "annex2a": ["annex 2a: consulting skills progression"],
    "annex1": ["annex 1: competency framework"],
    "role_profiles": ["role profile"]
    } 

In [0]:
def find_section_pages3(pages):
    """
    pages: dict of {page_num: text} (ALREADY sorted)
    CHANGED: dict of {page_num: ["text", [["table1"]], [["table2"]]]}
    Returns: dict {section_name: [page numbers]}
    """

    section_order = ["role_profiles", "annex4", "annex3", "annex2b", "annex2a", "annex1"]

    sorted_pages = sorted(pages.items(), key=lambda x: x[0])
    #sorted_pages = list(pages.items())  # keep structure consistent

    anchors = {}

    for sec in section_order:
        keywords = SECTION_KEYWORDS[sec]
        for page_num, text in sorted_pages:
            text_lower = text.lower()
            if any(kw in text_lower for kw in keywords):
                anchors.setdefault(sec, page_num)
                break

    anchored_sections = [(sec, anchors[sec]) for sec in section_order if sec in anchors]
    anchored_sections.sort(key=lambda x: x[1])

    section_pages = {sec: [] for sec in section_order}
    all_page_nums = [p for p, _ in sorted_pages]

    for i, (sec, start_page) in enumerate(anchored_sections):
        if i + 1 < len(anchored_sections):
            next_start = anchored_sections[i + 1][1]
        else:
            next_start = float("inf")

        section_pages[sec] = [p for p in all_page_nums if start_page <= p < next_start]

    return section_pages

In [0]:
sections = find_section_pages3(pages)
print(sections["annex2a"])

In [0]:
print(sections["annex4"])